## Severity Indicator - Casualty rate per 100 dwellings  and Civilian Injuries rate per 100 dwellings Calculation

To find the " Severity Indicator - Casualty rate per 100 dwellings" calculation from a CAD and RMS system by dissemination area (DA) in Barrie, we would need two sets of data:

Dwellings Data: Obtain the total number of dwellings in the area of interest. 
Casualty Data: Extract casualty data from the CAD/RMS system.

**Calculation:

Casualty Rate per 100 Dwellings: This is calculated using the formula:
* Casualty Rate per 100 Dwellings =(Total Number of Casualties/Total Number of Dwellings)×100


In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("../Cleaned_Dataset/new_DA.xlsx")
df.head()

,unique_Id,inci_no,inci_type,incident_category,Incident_Type_Description,Property_Loss_Value,Content_Loss_Value,Property_Value,Content_Value,Civilian_Fatal,...,Property_Damage_Category,Property_Damage_Description,Received_Datetime,Dispatched_Datetime,Arrival_Datetime,Cleared_Datetime,DAUID,Longitude,Latitude,DAUID_new
0,397231,20-0006971,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-10-31 05:21:00,2020-10-31 05:21:00,2020-10-31 05:27:00,2020-10-31 05:33:00,35431013.0,-79.697299,44.364071,35431014
1,397468,20-0007034,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,320,Multi-Unit Dwelling - Over 12 Units ...,2020-11-02 17:48:00,2020-11-02 17:48:00,2020-11-02 17:53:00,2020-11-02 18:22:00,35431321.0,-79.699290,44.354853,35430680
2,397549,20-0007052,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-03 15:16:00,2020-11-03 15:16:00,2020-11-03 15:22:00,2020-11-03 15:39:00,35431013.0,-79.697299,44.364071,35431014
3,398004,20-0007183,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,320,Multi-Unit Dwelling - Over 12 Units ...,2020-11-09 02:53:00,2020-11-09 02:53:00,2020-11-09 02:59:00,2020-11-09 03:06:00,35431321.0,-79.699290,44.354853,35430680
4,398196,20-0007238,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-11 00:38:00,2020-11-11 00:38:00,2020-11-11 00:45:00,2020-11-11 00:51:00,35431013.0,-79.697299,44.364071,35431014


## Add Dissemination Area

In [7]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import matplotlib.pyplot as plt

da_shapefile_path = '../Backend-app/lda_000b21a_e'

da_gdf = gpd.read_file(da_shapefile_path)

print(f"Dissemination Area CRS: {da_gdf.crs}")
print(f"Dissemination Area Columns: {da_gdf.columns}")

# Create a GeoDataFrame with your coordinates
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df['Longitude'], df['Latitude'])],
    crs="EPSG:4326"  # Assuming your coordinates are in WGS84
)

# Check the CRS of the point data
print(f"Original Points CRS: {gdf.crs}")

# Reproject point data to match the CRS of the dissemination area shapefile if needed
if gdf.crs != da_gdf.crs:
    gdf = gdf.to_crs(da_gdf.crs)

print(f"Reprojected Points CRS: {gdf.crs}")

result = gpd.sjoin(gdf, da_gdf, how="left", op="within")

print(f"Result Columns: {result.columns}")

# Add the new DAUID to the original dataframe
df['DAUID_new'] = result['DAUID_right']

print(df.head())

Dissemination Area CRS: EPSG:3347
Dissemination Area Columns: Index(['DAUID', 'DGUID', 'LANDAREA', 'PRUID', 'geometry'], dtype='object')
Original Points CRS: EPSG:4326
Reprojected Points CRS: EPSG:3347


/Users/balakumaransivarajan/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3466: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


Result Columns: Index(['unique_Id', 'inci_no', 'inci_type', 'incident_category',
       'Incident_Type_Description', 'Property_Loss_Value',
       'Content_Loss_Value', 'Property_Value', 'Content_Value',
       'Civilian_Fatal', 'Civilian_Injuries', 'prop_use',
       'Property_Damage_Category', 'Property_Damage_Description',
       'Received_Datetime', 'Dispatched_Datetime', 'Arrival_Datetime',
       'Cleared_Datetime', 'DAUID_left', 'Longitude', 'Latitude', 'geometry',
       'index_right', 'DAUID_right', 'DGUID', 'LANDAREA', 'PRUID'],
      dtype='object')
   unique_Id     inci_no  inci_type          incident_category  \
0     397231  20-0006971         89  I                           
1     397468  20-0007034         89  I                           
2     397549  20-0007052         89  I                           
3     398004  20-0007183         89  I                           
4     398196  20-0007238         89  I                           

                           Incident_

## Finding Casuality rate per incident Civilian Injuries rate per incident by DA

In [7]:
# Group by 'DAUID_new' and sum the 'Civilian_Fatal' and 'Civilian_Injuries' columns
civilian_fatal_count = df.groupby('DAUID_new')['Civilian_Fatal'].sum().reset_index()
civilian_injuries_count = df.groupby('DAUID_new')['Civilian_Injuries'].sum().reset_index()

# Group by 'DAUID_new' and count the number of incidents (each row is one incident)
total_fire_incidents = df.groupby('DAUID_new').size().reset_index(name='Fire_Incidents')

# Merge the three DataFrames on 'DAUID_new'
merged_df = pd.merge(civilian_fatal_count, total_fire_incidents, on='DAUID_new')
merged_df = pd.merge(merged_df, civilian_injuries_count, on='DAUID_new')

# Calculate the casualty rate per incident
merged_df['Casualty_Rate_Per_Incident'] = (merged_df['Civilian_Fatal'] / merged_df['Fire_Incidents']).round(2)

# Calculate the civilian injuries rate per incident
merged_df['Injuries_Rate_Per_Incident'] = (merged_df['Civilian_Injuries'] / merged_df['Fire_Incidents']).round(2)

# Rename the columns for clarity
merged_df.rename(columns={'DAUID_new': 'DAUID', 'Civilian_Fatal': 'Total_Civilian_Fatalities', 'Civilian_Injuries': 'Total_Civilian_Injuries'}, inplace=True)

# Display the result
print(merged_df)

# Save the result to a CSV file
merged_df.to_csv('../cleaned_dataset/casualty_and_injuries_rate_per_incident_by_dguid.csv', index=False, columns=['DAUID', 'Total_Civilian_Fatalities', 'Fire_Incidents', 'Casualty_Rate_Per_Incident', 'Total_Civilian_Injuries', 'Injuries_Rate_Per_Incident'])

print("Data saved to ../cleaned_dataset/casualty_and_injuries_rate_per_incident_by_dguid.csv")

        DAUID  Total_Civilian_Fatalities  Fire_Incidents  \
0    35430639                          0               7   
1    35430647                          0             130   
2    35430648                          0              87   
3    35430649                          0              70   
4    35430650                          0              10   
..        ...                        ...             ...   
241  35431371                          0              38   
242  35431372                          0              75   
243  35431373                          0             270   
244  35431377                          0              35   
245  35431380                          0             327   

     Total_Civilian_Injuries  Casualty_Rate_Per_Incident  \
0                          0                         0.0   
1                          0                         0.0   
2                          0                         0.0   
3                          0           